# Notebook 01: Synthetic Ellipse Case Study

**Obstacle-Aware Clustering for Geographic Data**

This notebook demonstrates why standard k-Means fails in the presence of geographic obstacles and how a loop-aware arc-length parameter corrects the problem. We work through a fully controlled synthetic example using an elliptical obstacle before applying the method to real-world data in later notebooks.

---

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

# Our custom package
from obstacle_clustering import (
    EllipseBoundary, ObstacleKMeans,
    loop_aware_distance, plot_clusters, plot_comparison
)

# Reproducibility
SEED = 23
np.random.seed(SEED)

# Plot style
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 2. Defining the Obstacle

We define our obstacle as the level set of a 2D Gaussian function at threshold $c$:

$$A \cdot \exp\!\left(-\frac{(x-h)^2}{2\sigma_x^2} - \frac{(y-k)^2}{2\sigma_y^2}\right) = c$$

Solving for the boundary gives a parametric ellipse:

$$x(t) = \sigma_x \sqrt{-2\ln(c/A)}\,\cos(t) + h, \qquad y(t) = \sigma_y \sqrt{-2\ln(c/A)}\,\sin(t) + k$$

for $t \in [0, 2\pi]$. The parameters $\sigma_x$ and $\sigma_y$ control the aspect ratio, while $(h, k)$ sets the center.

In [ ]:
# Define the ellipse boundary
boundary = EllipseBoundary(
    sigma_x=3.0, sigma_y=0.6,
    h=6.0, k=0.0,
    c=0.01, A=1.0
)

# Compute and display total arc length
L = boundary.total_arc_length()
print(f'Total arc length of ellipse: {L:.4f}')
print(f'Sqrt term (semi-axis scale): {boundary.sqrt_term:.4f}')

# Plot the ellipse
boundary_pts = boundary.sample_boundary(500)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(boundary_pts[:, 0], boundary_pts[:, 1], 'k-', linewidth=2, label='Ellipse obstacle')
ax.fill(boundary_pts[:, 0], boundary_pts[:, 1], alpha=0.15, color='gray')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Elliptical Obstacle Boundary')
ax.set_aspect('equal')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Generating Synthetic Data

We generate three clusters of points around the ellipse, each with distinct spatial positions and attribute values. For each cluster, we sample:

- **Geographic coordinates** $(x, y)$: uniform base + Gaussian noise, positioned around different parts of the ellipse
- **Housing price** $h$: drawn from cluster-specific Gaussian distributions (simulating an attribute like property value)
- **Favorability rating** $r$: drawn from cluster-specific Gaussians

The clusters are placed so that two of them sit on opposite sides of the ellipse but are relatively close in Euclidean distance. This is the scenario where standard k-Means will fail.

In [ ]:
def generate_synthetic_data(k=3, n=7, seed=23):
    """Generate synthetic data around an elliptical obstacle."""
    np.random.seed(seed)
    
    cluster_params = [
        {'x_range': (7, 11), 'y_range': (-4, -2), 'x_std': 2, 'y_std': 0.2},
        {'x_range': (0, 2.5), 'y_range': (-3, -2), 'x_std': 2, 'y_std': 0.2},
        {'x_range': (6, 10), 'y_range': (3, 6), 'x_std': 2, 'y_std': 0.2},
    ]
    housing_params = [
        {'mean': 500000, 'std': 50000},
        {'mean': 200000, 'std': 50000},
        {'mean': 350000, 'std': 50000},
    ]
    rating_params = [
        {'mean': 0.3, 'std': 0.1},
        {'mean': 0.9, 'std': 0.1},
        {'mean': 0.6, 'std': 0.1},
    ]
    
    all_points = []
    true_labels = []
    
    for i in range(k):
        p = cluster_params[i]
        x = np.random.uniform(*p['x_range'], n) + np.random.normal(0, p['x_std'], n)
        y = np.random.uniform(*p['y_range'], n) + np.random.normal(0, p['y_std'], n)
        h = np.random.normal(housing_params[i]['mean'], housing_params[i]['std'], n)
        r = np.random.normal(rating_params[i]['mean'], rating_params[i]['std'], n)
        all_points.append(np.column_stack([x, y, h, r]))
        true_labels.extend([i] * n)
    
    return np.vstack(all_points), np.array(true_labels)


# Generate data
raw_data, true_labels = generate_synthetic_data()
print(f'Generated {len(raw_data)} data points across 3 clusters')
print(f'Columns: x, y, housing_price, rating')
print(f'Shape: {raw_data.shape}')

In [ ]:
# Visualize raw data with true cluster labels and the ellipse
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#e74c3c', '#3498db', '#2ecc71']
labels = ['Cluster 1 (bottom-right)', 'Cluster 2 (bottom-left)', 'Cluster 3 (top)']

for i in range(3):
    mask = true_labels == i
    ax.scatter(raw_data[mask, 0], raw_data[mask, 1], c=colors[i],
              s=80, edgecolors='white', linewidth=0.5, label=labels[i], zorder=3)

# Ellipse
ax.plot(boundary_pts[:, 0], boundary_pts[:, 1], 'k--', linewidth=1.5, label='Obstacle')
ax.fill(boundary_pts[:, 0], boundary_pts[:, 1], alpha=0.1, color='gray')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Synthetic Data: True Cluster Assignments')
ax.legend(loc='upper left')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Failure of Standard k-Means

Standard k-Means uses Euclidean distance in $(x, y)$ to assign points to clusters. It has no concept of the obstacle — it treats the space as unobstructed. As a result, points on opposite sides of the ellipse may be grouped together simply because their straight-line distance is small.

Let's see this in action.

In [ ]:
# Run standard k-Means on (x, y) only
xy_data = raw_data[:, :2]

kmeans_standard = KMeans(n_clusters=3, random_state=SEED, n_init=10)
labels_standard = kmeans_standard.fit_predict(xy_data)

# Plot standard k-Means result
fig, ax = plt.subplots(figsize=(10, 6))
cmap = cm.get_cmap('tab10', 3)

for i in range(3):
    mask = labels_standard == i
    ax.scatter(xy_data[mask, 0], xy_data[mask, 1], c=[cmap(i)],
              s=80, edgecolors='white', linewidth=0.5, label=f'Cluster {i+1}', zorder=3)

# Centroids
ax.scatter(kmeans_standard.cluster_centers_[:, 0], kmeans_standard.cluster_centers_[:, 1],
           c='red', marker='X', s=200, edgecolors='black', linewidth=1.5, zorder=4, label='Centroids')

# Ellipse
ax.plot(boundary_pts[:, 0], boundary_pts[:, 1], 'k--', linewidth=1.5, label='Obstacle')
ax.fill(boundary_pts[:, 0], boundary_pts[:, 1], alpha=0.1, color='gray')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Standard k-Means: Clusters Ignore the Obstacle')
ax.legend(loc='upper left')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Notice: standard k-Means may group points from opposite sides of the ellipse')
print('because it only considers straight-line distance, ignoring the barrier.')

## 5. The Arc-Length Parameter $s$

To encode obstacle awareness, we introduce a normalized arc-length parameter $s \in [0, 1]$ for each data point:

1. **Project** the point onto the nearest location on the ellipse boundary
2. **Compute** the arc length from a reference point ($t=0$) to the projection
3. **Normalize** by the total perimeter: $s = \text{arc length} \,/\, L$

The key insight is the **loop-aware distance**:

$$d_s(s_1, s_2) = \min\big(|s_1 - s_2|,\; 1 - |s_1 - s_2|\big)$$

This accounts for the closed-loop topology. Points with $s = 0.05$ and $s = 0.95$ are actually close (distance = 0.10), not far apart (0.90).

In [ ]:
# Project each data point onto the ellipse and compute s
t_values = []
s_values = []

for x_p, y_p in raw_data[:, :2]:
    t_closest, s_closest = boundary.project_point(x_p, y_p)
    t_values.append(t_closest)
    s_values.append(s_closest)

t_values = np.array(t_values)
s_values = np.array(s_values)

print('Arc-length values (s) for each point:')
for i, (x, y, s) in enumerate(zip(raw_data[:, 0], raw_data[:, 1], s_values)):
    print(f'  Point {i:2d}: ({x:6.2f}, {y:6.2f})  →  s = {s:.4f}')

In [ ]:
# Visualize points with their s values labeled
fig, ax = plt.subplots(figsize=(10, 6))

scatter = ax.scatter(raw_data[:, 0], raw_data[:, 1], c=s_values, cmap='hsv',
                     s=80, edgecolors='black', linewidth=0.5, zorder=3)
plt.colorbar(scatter, ax=ax, label='Normalized arc-length $s$')

# Label each point with its s value
for i, (x, y, s) in enumerate(zip(raw_data[:, 0], raw_data[:, 1], s_values)):
    ax.annotate(f'{s:.2f}', (x, y), textcoords='offset points',
               xytext=(5, 5), fontsize=8, color='darkred')

# Ellipse
ax.plot(boundary_pts[:, 0], boundary_pts[:, 1], 'k--', linewidth=1.5)
ax.fill(boundary_pts[:, 0], boundary_pts[:, 1], alpha=0.1, color='gray')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Data Points Colored by Arc-Length Parameter $s$')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate loop-aware distance
print('Loop-aware distance examples:')
print(f'  d(0.05, 0.95) = {loop_aware_distance(0.05, 0.95):.2f}  (close — wraps around)')
print(f'  d(0.05, 0.50) = {loop_aware_distance(0.05, 0.50):.2f}  (far — opposite sides)')
print(f'  d(0.30, 0.70) = {loop_aware_distance(0.30, 0.70):.2f}  (moderately far)')
print(f'  d(0.10, 0.20) = {loop_aware_distance(0.10, 0.20):.2f}  (close — same region)')

## 6. Feature Engineering

We now construct the full feature vector for each data point. All features are normalized to $[0, 1]$ so that no single dimension dominates the distance calculation:

$$\mathbf{x}_i = (x_i^{\text{scaled}},\; y_i^{\text{scaled}},\; s_i,\; h_i^{\text{scaled}},\; r_i^{\text{scaled}})$$

Note that $s$ is already in $[0, 1]$ by construction.

In [ ]:
# Normalize features
scaler_xy = MinMaxScaler()
xy_scaled = scaler_xy.fit_transform(raw_data[:, :2])

scaler_h = MinMaxScaler()
h_scaled = scaler_h.fit_transform(raw_data[:, 2:3])

# r values are already roughly in [0, 1], but normalize for consistency
scaler_r = MinMaxScaler()
r_scaled = scaler_r.fit_transform(raw_data[:, 3:4])

# Combine into feature matrix: [x, y, s, h, r]
X = np.column_stack([xy_scaled, s_values, h_scaled.ravel(), r_scaled.ravel()])

print(f'Feature matrix shape: {X.shape}')
print(f'Features: x_scaled, y_scaled, s, h_scaled, r_scaled')
print(f'\nFirst 5 rows:')
print(np.array2string(X[:5], precision=4, suppress_small=True))

## 7. Obstacle-Aware k-Means

Our modified k-Means uses a weighted composite distance:

$$d^2(\mathbf{x}_i, \mathbf{c}_j) = \alpha^2 \|(x_i, y_i) - (c_{jx}, c_{jy})\|^2 + \beta^2\, d_s(s_i, s_{c_j})^2 + \gamma^2\big((h_i - h_{c_j})^2 + (r_i - r_{c_j})^2\big)$$

The three weight parameters control the balance between:
- $\alpha$: geographic proximity (Euclidean in $x, y$)
- $\beta$: position along the obstacle (loop-aware in $s$)
- $\gamma$: attribute similarity (housing price, rating)

The centroid update for $s$ uses **boundary-aware projection**: instead of naively averaging $s$ values (which would fail on a circle), we map cluster members back to boundary coordinates, average those, and re-project onto the boundary.

In [ ]:
# Run obstacle-aware k-Means
model = ObstacleKMeans(
    k=3,
    boundary=boundary,
    alpha=1.0,
    beta=1.0,
    gamma=1.0,
    random_state=35,
    n_attr=2  # h and r are the attribute features
)
model.fit(X, t_data=t_values)

labels_obstacle = model.labels_

print(f'Converged in {model.n_iter_} iterations')
print(f'Within-cluster distortion (rho_bar): {model.rho_bar_:.6f}')
print(f'Cluster assignments: {labels_obstacle}')

## 8. Comparison: Standard vs. Obstacle-Aware

Now we can directly compare the two approaches side by side. The key question: does the obstacle-aware version produce clusters that respect the boundary geometry?

In [ ]:
# Side-by-side comparison using original (unscaled) coordinates
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Standard k-Means ---
ax = axes[0]
cmap = cm.get_cmap('tab10', 3)
for i in range(3):
    mask = labels_standard == i
    ax.scatter(raw_data[mask, 0], raw_data[mask, 1], c=[cmap(i)],
              s=80, edgecolors='white', linewidth=0.5, label=f'Cluster {i+1}', zorder=3)
ax.scatter(kmeans_standard.cluster_centers_[:, 0], kmeans_standard.cluster_centers_[:, 1],
           c='red', marker='X', s=200, edgecolors='black', linewidth=1.5, zorder=4)
ax.plot(boundary_pts[:, 0], boundary_pts[:, 1], 'k--', linewidth=1.5)
ax.fill(boundary_pts[:, 0], boundary_pts[:, 1], alpha=0.1, color='gray')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Standard k-Means\n(Euclidean distance only)', fontsize=13)
ax.legend(loc='upper left', fontsize=9)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# --- Obstacle-Aware k-Means ---
ax = axes[1]
for i in range(3):
    mask = labels_obstacle == i
    ax.scatter(raw_data[mask, 0], raw_data[mask, 1], c=[cmap(i)],
              s=80, edgecolors='white', linewidth=0.5, label=f'Cluster {i+1}', zorder=3)

# Compute centroids in original coordinates for plotting
for i in range(3):
    mask = labels_obstacle == i
    cx = np.mean(raw_data[mask, 0])
    cy = np.mean(raw_data[mask, 1])
    ax.scatter(cx, cy, c='red', marker='X', s=200, edgecolors='black', linewidth=1.5, zorder=4)

ax.plot(boundary_pts[:, 0], boundary_pts[:, 1], 'k--', linewidth=1.5)
ax.fill(boundary_pts[:, 0], boundary_pts[:, 1], alpha=0.1, color='gray')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Obstacle-Aware k-Means\n(with arc-length parameter $s$)', fontsize=13)
ax.legend(loc='upper left', fontsize=9)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

plt.suptitle('Effect of the Arc-Length Parameter on Clustering', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 9. Cluster Summaries

Let's examine the composition of each obstacle-aware cluster to verify that the algorithm produces groups that differ meaningfully in both spatial position and attributes.

In [ ]:
# Summarize each cluster
feature_names = ['x_scaled', 'y_scaled', 's', 'h_scaled', 'r_scaled']
summaries = model.get_cluster_summary(X, feature_names=feature_names)

for s in summaries:
    print(f"\nCluster {s['cluster'] + 1} ({s['size']} points):")
    print(f"  Centroid: {[f'{v:.3f}' for v in s['centroid']]}")
    for fname, stats in s['features'].items():
        print(f"  {fname:12s}: mean={stats['mean']:.3f}, std={stats['std']:.3f}")

## 10. Discussion

This synthetic case study demonstrates the core ideas:

1. **Standard k-Means fails** when obstacles separate data points that are close in Euclidean distance but far apart by any realistic path.

2. **The arc-length parameter** $s$ encodes position along the obstacle boundary. Its loop-aware distance respects the closed-curve topology.

3. **The modified k-Means** combines Euclidean geography, loop-aware arc-length, and attribute features in a single weighted distance. Centroid updates for $s$ use boundary-aware projection rather than naive averaging.

4. **The approach is general**: it does not depend on the obstacle being an ellipse. Any smooth closed curve that admits a parameterization will work — including cubic splines fitted to real-world boundaries.

### Next Steps

In **Notebook 02**, we extract the Lake Tahoe boundary using ArcGIS and cubic splines. In **Notebook 03**, we apply the same algorithm to wildfire ignition data, where the lake is the obstacle and fire behavior attributes replace housing prices.